# 04 Bibliographic Coupling Network

In [1]:
from pathlib import Path
import pickle
import time
import numpy as np
import pandas as pd
from datasketch import MinHash, MinHashLSH

In [2]:
cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name == 'notebooks' else cwd
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
DIRECT_DATA_DIR = PROJECT_ROOT / 'outputs' / '02_direct_citation_network' / 'data'
NOTEBOOK_OUTPUT_DIR = PROJECT_ROOT / 'outputs' / '04_bibliographic_coupling_network'
FIGURE_DIR = NOTEBOOK_OUTPUT_DIR / 'figures'
TABLE_DIR = NOTEBOOK_OUTPUT_DIR / 'tables'
DATA_OUTPUT_DIR = NOTEBOOK_OUTPUT_DIR / 'data'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)
DATA_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Load Final Citation Data

In [3]:
citation_edges = pd.read_pickle(PROCESSED_DIR / 'citation_edges_final.pkl').rename(columns={'citing_paper_id': 'citing', 'cited_paper_id': 'cited'})[['citing', 'cited']].copy()
papers = pd.read_feather(DIRECT_DATA_DIR / 'papers_with_degrees.feather')
citation_edges['citing'] = citation_edges['citing'].astype(str)
citation_edges['cited'] = citation_edges['cited'].astype(str)
papers['id'] = papers['id'].astype(str)
print('Citation edges:', citation_edges.shape)
print('Papers:', papers.shape)
display(citation_edges.head())

Citation edges: (18904534, 2)
Papers: (3026412, 32)


,citing,cited
0,W2572734975,W238389808
1,W2572734975,W1986370907
2,W2572734975,W1997685381
4,W2572734975,W2791342857
5,W2572734975,W3013283542


## 2. Create Citing-Paper Profiles

In [4]:
start = time.time()
citing_profiles = citation_edges.groupby('citing')['cited'].agg(lambda x: set(x)).reset_index(name='reference_set')
citing_profiles['n_references_internal'] = citing_profiles['reference_set'].apply(len)
elapsed = time.time() - start
print('Created citing profiles:', citing_profiles.shape)
print('Elapsed seconds:', round(elapsed, 2))
citing_profiles.head()

Created citing profiles: (2733997, 3)
Elapsed seconds: 27.8


,citing,reference_set,n_references_internal
0,W1000022707,"{W2086178947, W2069792094}",2
1,W100007563,"{W2078435329, W2115309705, W1964167056, W20890...",4
2,W100008278,"{W1972013208, W1976893898, W1984588326, W21093...",15
3,W100009080,"{W2160170050, W2133109597, W1555476679, W15535...",5
4,W1000189836,{W2088278063},1


In [5]:
citing_profiles['n_references_internal'].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99])

count    2.733997e+06
mean     6.914614e+00
std      6.771499e+00
min      1.000000e+00
25%      3.000000e+00
50%      5.000000e+00
75%      9.000000e+00
90%      1.500000e+01
95%      1.900000e+01
99%      2.900000e+01
max      1.090000e+03
Name: n_references_internal, dtype: float64

## 3. Filter Citing Papers for Bibliographic Coupling Construction

In [6]:
MIN_INTERNAL_REFERENCES = 5
citing_profiles_filtered = citing_profiles.loc[citing_profiles['n_references_internal'] >= MIN_INTERNAL_REFERENCES].copy().reset_index(drop=True)
print('Original citing papers:', len(citing_profiles))
print('Filtered citing papers:', len(citing_profiles_filtered))
print('Share kept:', round(len(citing_profiles_filtered) / len(citing_profiles) * 100, 2), '%')
citing_profiles_filtered.head()

Original citing papers: 2733997
Filtered citing papers: 1514100
Share kept: 55.38 %


,citing,reference_set,n_references_internal
0,W100008278,"{W1972013208, W1976893898, W1984588326, W21093...",15
1,W100009080,"{W2160170050, W2133109597, W1555476679, W15535...",5
2,W100019873,"{W2065592694, W2069284715, W2090758519, W20015...",5
3,W100029996,"{W2033570535, W1565831494, W2141147872, W20764...",14
4,W1000312060,"{W2115725211, W2122633938, W1996832037, W20277...",6


## 4. Build MinHash Signatures

In [7]:
NUM_PERM = 128

def make_minhash(values, num_perm=NUM_PERM):
    m = MinHash(num_perm=num_perm)
    for value in values:
        m.update(str(value).encode('utf-8'))
    return m

In [8]:
start = time.time()
minhashes = {}
for idx, row in citing_profiles_filtered.iterrows():
    paper_id = row['citing']
    minhashes[paper_id] = make_minhash(row['reference_set'])
elapsed = time.time() - start
print('Created MinHash signatures:', len(minhashes))
print('Elapsed seconds:', round(elapsed, 2))

Created MinHash signatures: 1514100
Elapsed seconds: 255.76


## 5. Build LSH Index

In [9]:
LSH_THRESHOLD = 0.25
start = time.time()
lsh = MinHashLSH(threshold=LSH_THRESHOLD, num_perm=NUM_PERM)
for paper_id, minhash in minhashes.items():
    lsh.insert(paper_id, minhash)
elapsed = time.time() - start
print('Inserted signatures into LSH:', len(minhashes))
print('LSH threshold:', LSH_THRESHOLD)
print('Elapsed seconds:', round(elapsed, 2))

Inserted signatures into LSH: 1514100
LSH threshold: 0.25
Elapsed seconds: 118.93


## 6. Generate Candidate Bibliographic-Coupling Pairs

In [10]:
reference_sets = dict(zip(citing_profiles_filtered['citing'], citing_profiles_filtered['reference_set']))
n_references_internal = dict(zip(citing_profiles_filtered['citing'], citing_profiles_filtered['n_references_internal']))
print('Prepared lookup dictionaries:', len(reference_sets))

Prepared lookup dictionaries: 1514100


In [11]:
MIN_SHARED_REFERENCES = 3
start = time.time()
pairs = []
seen_pairs = set()
paper_ids = list(reference_sets.keys())
n_papers = len(paper_ids)
for idx, paper_i in enumerate(paper_ids, start=1):
    candidate_ids = lsh.query(minhashes[paper_i])
    refs_i = reference_sets[paper_i]
    for paper_j in candidate_ids:
        if paper_i == paper_j:
            continue
        if paper_i < paper_j:
            pair_key = (paper_i, paper_j)
        else:
            pair_key = (paper_j, paper_i)
        if pair_key in seen_pairs:
            continue
        seen_pairs.add(pair_key)
        refs_j = reference_sets[paper_j]
        shared_references = len(refs_i.intersection(refs_j))
        if shared_references < MIN_SHARED_REFERENCES:
            continue
        union_size = len(refs_i) + len(refs_j) - shared_references
        jaccard_reference_sets = shared_references / union_size if union_size > 0 else 0.0
        paper_a, paper_b = pair_key
        pairs.append({'shared_references': shared_references, 'jaccard_reference_sets': jaccard_reference_sets, 'paper_i_n_references_internal': n_references_internal[paper_a], 'paper_j_n_references_internal': n_references_internal[paper_b], 'paper_i': paper_a, 'paper_j': paper_b})
elapsed = time.time() - start
bib_coupling_candidate_edges = pd.DataFrame(pairs)
print('Candidate bibliographic-coupling edges:', bib_coupling_candidate_edges.shape)
print('Elapsed seconds:', round(elapsed, 2))
bib_coupling_candidate_edges.head()

Candidate bibliographic-coupling edges: (9079863, 6)
Elapsed seconds: 176.19


,shared_references,jaccard_reference_sets,paper_i_n_references_internal,paper_j_n_references_internal,paper_i,paper_j
0,5,0.208333,15,14,W100008278,W659058298
1,5,0.108696,15,36,W100008278,W2440218450
2,3,0.085714,15,23,W100008278,W2024981255
3,3,0.085714,15,23,W100008278,W2586669864
4,6,0.133333,15,36,W100008278,W2021778777


In [12]:
bib_coupling_candidate_edges['shared_references'].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99])

count    9.079863e+06
mean     3.725101e+00
std      1.398697e+00
min      3.000000e+00
25%      3.000000e+00
50%      3.000000e+00
75%      4.000000e+00
90%      5.000000e+00
95%      6.000000e+00
99%      9.000000e+00
max      1.950000e+02
Name: shared_references, dtype: float64

## 7. Create Sparse Top-K Bibliographic Coupling Network

In [13]:
TOP_K = 20
edges_forward = bib_coupling_candidate_edges.rename(columns={'paper_i': 'source', 'paper_j': 'target'})
edges_reverse = bib_coupling_candidate_edges.rename(columns={'paper_j': 'source', 'paper_i': 'target'})
edges_bidirectional = pd.concat([edges_forward, edges_reverse], ignore_index=True)
edges_bidirectional = edges_bidirectional.sort_values(['source', 'shared_references', 'jaccard_reference_sets'], ascending=[True, False, False])
topk_directed = edges_bidirectional.groupby('source', group_keys=False).head(TOP_K).reset_index(drop=True)
print('Directed top-k edges:', topk_directed.shape)
topk_directed.head()

Directed top-k edges: (6312746, 6)


,shared_references,jaccard_reference_sets,paper_i_n_references_internal,paper_j_n_references_internal,source,target
0,7,0.189189,15,29,W100008278,W2026051658
1,7,0.148936,15,39,W100008278,W387768486
2,6,0.250000,15,15,W100008278,W1015062377
3,6,0.133333,15,36,W100008278,W2021778777
4,5,0.208333,15,14,W100008278,W659058298


In [14]:
topk_directed['paper_i'] = topk_directed[['source', 'target']].min(axis=1)
topk_directed['paper_j'] = topk_directed[['source', 'target']].max(axis=1)
bib_coupling_topk_edges = topk_directed.drop(columns=['source', 'target']).drop_duplicates(subset=['paper_i', 'paper_j']).sort_values(['shared_references', 'jaccard_reference_sets'], ascending=False).reset_index(drop=True)
print('Undirected top-k bibliographic-coupling edges:', bib_coupling_topk_edges.shape)
bib_coupling_topk_edges.head()

Undirected top-k bibliographic-coupling edges: (3934046, 6)


,shared_references,jaccard_reference_sets,paper_i_n_references_internal,paper_j_n_references_internal,paper_i,paper_j
0,195,0.550847,203,346,W1985697265,W2904923893
1,146,1.000000,146,146,W4293464704,W4293464712
2,145,0.993151,145,146,W4293464697,W4293464712
3,145,0.993151,145,146,W4293464697,W4293464704
4,142,0.972603,146,142,W4293464704,W4293464716


## 9. Network Summary

In [15]:
bib_coupling_summary = pd.Series({'n_direct_citation_edges': len(citation_edges), 'n_citing_papers': len(citing_profiles), 'min_internal_references': MIN_INTERNAL_REFERENCES, 'n_citing_papers_after_filtering': len(citing_profiles_filtered), 'num_perm': NUM_PERM, 'lsh_threshold': LSH_THRESHOLD, 'min_shared_references': MIN_SHARED_REFERENCES, 'n_candidate_edges': len(bib_coupling_candidate_edges), 'top_k': TOP_K, 'n_topk_edges': len(bib_coupling_topk_edges), 'n_topk_nodes': len(set(bib_coupling_topk_edges['paper_i']).union(set(bib_coupling_topk_edges['paper_j']))), 'mean_shared_references': round(bib_coupling_topk_edges['shared_references'].mean(), 3), 'median_shared_references': round(bib_coupling_topk_edges['shared_references'].median(), 3), 'max_shared_references': int(bib_coupling_topk_edges['shared_references'].max())})
bib_coupling_summary

n_direct_citation_edges            1.890453e+07
n_citing_papers                    2.733997e+06
min_internal_references            5.000000e+00
n_citing_papers_after_filtering    1.514100e+06
num_perm                           1.280000e+02
lsh_threshold                      2.500000e-01
min_shared_references              3.000000e+00
n_candidate_edges                  9.079863e+06
top_k                              2.000000e+01
n_topk_edges                       3.934046e+06
n_topk_nodes                       9.371610e+05
mean_shared_references             4.253000e+00
median_shared_references           4.000000e+00
max_shared_references              1.950000e+02
dtype: float64

## Save Outputs

In [16]:
def save_pickle(obj, path):
    with open(path, 'wb') as f:
        pickle.dump(obj, f)

In [17]:
bib_coupling_topk_path = DATA_OUTPUT_DIR / 'net_bib_coupling_topk.pkl'
save_pickle(bib_coupling_topk_edges, bib_coupling_topk_path)
print('Saved:', bib_coupling_topk_path)

Saved: c:\Users\blaze\OneDrive\Documents\GitHub\citation-gender-imbalance\outputs\04_bibliographic_coupling_network\data\net_bib_coupling_topk.pkl


In [18]:
bib_coupling_summary.to_csv(TABLE_DIR / 'net_bib_coupling_summary.csv')
print('Saved:', TABLE_DIR / 'net_bib_coupling_summary.csv')

Saved: c:\Users\blaze\OneDrive\Documents\GitHub\citation-gender-imbalance\outputs\04_bibliographic_coupling_network\tables\net_bib_coupling_summary.csv
